# Notebook Overview

This notebook demonstrates a complete pipeline for predicting the compressive strength of concrete/materials using a pre-trained XGBoost model. The workflow includes:

**Data Fetching**

1. Pulling material data from a Databricks SQL warehouse.

2. Ensuring numeric consistency and preparing features for modeling.

3. Feature Engineering & Preprocessing

4. Computing derived features such as Water-to-Cement Ratio (WCR) and Log(Age).

5. Handling missing values and ensuring the input aligns with the model’s expected features.

**Model Inference**

1. Loading a pre-trained XGBoost model.

2. Predicting material strength for all samples.

3. Applying inverse transformations if features were log-transformed.

**Health Report Generation**

Producing a material strength inference report similar to a production monitoring dashboard.


In [1]:
from google.colab import files
uploaded = files.upload()

Saving xgboost_material_strength_model.pkl to xgboost_material_strength_model.pkl


**Data Confirmation**

In [40]:
import requests
import json
import pandas as pd
import time

# Databricks API config
DATABRICKS_INSTANCE = "https://dbc-c4b69e1c-9eb0.cloud.databricks.com"
TOKEN = "Token has been removed after running this notebook"
SQL_ENDPOINT_ID = "120960815ade2435"

query = "SELECT * FROM `processed_material_strength_dataset`"

# 1️⃣ Submit the query
url_submit = f"{DATABRICKS_INSTANCE}/api/2.0/sql/statements/"
headers = {
    "Authorization": f"Bearer {TOKEN}",
    "Content-Type": "application/json"
}
data = {
    "statement": query,
    "warehouse_id": SQL_ENDPOINT_ID
}

response = requests.post(url_submit, headers=headers, data=json.dumps(data))
res_json = response.json()
statement_id = res_json['statement_id']
print(f"Query submitted. Statement ID: {statement_id}")

# 2️⃣ Poll for results
url_status = f"{DATABRICKS_INSTANCE}/api/2.0/sql/statements/{statement_id}"
while True:
    resp = requests.get(url_status, headers=headers)
    status_json = resp.json()
    state = status_json['status']['state']

    if state == 'SUCCEEDED':
        print("Query succeeded!")
        break
    elif state in ['FAILED', 'CANCELED']:
        raise Exception(f"Query {state}: {status_json}")
    else:
        print(f"Query running... current state: {state}")
        time.sleep(1)

# 3️⃣ Fetch results depending on the response format
result = status_json.get('result')

import pandas as pd

# Extract column names
columns = [col['name'] for col in status_json['manifest']['schema']['columns']]

# Extract row data
rows = status_json['result']['data_array']

# Convert to DataFrame
df = pd.DataFrame(rows, columns=columns)

# convert numeric columns from string to float/int
for col in df.columns:
    df[col] = pd.to_numeric(df[col])


print(df.head())

Query submitted. Statement ID: 01f10417-4ebd-1e5d-a367-37490dec5585
Query succeeded!
   Cement   Slag  FlyAsh  Water  Superplasticizer  CoarseAgg  FineAgg  Age
0   540.0    0.0     0.0  162.0               2.5     1040.0    676.0   28
1   540.0    0.0     0.0  162.0               2.5     1055.0    676.0   28
2   332.5  142.5     0.0  228.0               0.0      932.0    594.0  270
3   332.5  142.5     0.0  228.0               0.0      932.0    594.0  365
4   198.6  132.4     0.0  192.0               0.0      978.4    825.5  360


**Pipelines**

In [42]:
from dataclasses import dataclass, field
from typing import List
import logging
import sys
import requests
import time
import pandas as pd
import joblib
import numpy as np


# ---------------- LOGGING ----------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)


# ---------------- CONFIG ----------------
@dataclass
class Config:
    DATABRICKS_INSTANCE: str = "https://dbc-c4b69e1c-9eb0.cloud.databricks.com"
    TOKEN: str = "Token has been removed after running this notebook"
    SQL_ENDPOINT_ID: str = "120960815ade2435"

    SOURCE_TABLE: str = "processed_material_strength_dataset"
    MODEL_PATH: str = "xgboost_material_strength_model.pkl"

    DROP_COLUMNS: List[str] = field(default_factory=lambda: [
        "Strength", "Age", "FlyAsh", "Age_x_WCR", "TotalAgg"
    ])


# ---------------- DATA FETCH ----------------
def fetch_databricks_data(cfg: Config) -> pd.DataFrame:
    query = f"SELECT * FROM {cfg.SOURCE_TABLE}"
    url_submit = f"{cfg.DATABRICKS_INSTANCE}/api/2.0/sql/statements/"
    headers = {"Authorization": f"Bearer {cfg.TOKEN}", "Content-Type": "application/json"}
    payload = {"statement": query, "warehouse_id": cfg.SQL_ENDPOINT_ID}

    r = requests.post(url_submit, headers=headers, json=payload)
    r.raise_for_status()
    statement_id = r.json()["statement_id"]
    logger.info(f"Query submitted: {statement_id}")

    status_url = f"{cfg.DATABRICKS_INSTANCE}/api/2.0/sql/statements/{statement_id}"
    while True:
        resp = requests.get(status_url, headers=headers).json()
        state = resp["status"]["state"]
        if state == "SUCCEEDED":
            break
        elif state in ["FAILED", "CANCELED"]:
            raise RuntimeError(resp)
        time.sleep(1)

    columns = [c["name"] for c in resp["manifest"]["schema"]["columns"]]
    rows = resp["result"]["data_array"]
    df = pd.DataFrame(rows, columns=columns)

    # Convert all columns to numeric
    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df

# ---------------- PREPROCESS ----------------
def preprocess_features(df: pd.DataFrame, model) -> pd.DataFrame:
    df = df.copy()

    # Drop columns that are not part of model features
    df = df.drop(columns=["Strength", "FlyAsh", "Age_x_WCR", "TotalAgg"], errors="ignore")

    # Create Log_Age from Age if it exists, else create Log_Age as 0
    if "Age" in df.columns:
        df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
        df["Log_Age"] = np.log1p(df["Age"])  # log1p = log(1 + x) to handle Age=0
    else:
        logger.warning("Missing 'Age' column → creating 'Log_Age' with zeros")
        df["Log_Age"] = 0.0

    # Create WCR from Water / Cement if not present
    if "WCR" not in df.columns:
        if "Water" in df.columns and "Cement" in df.columns:
            df["WCR"] = df["Water"] / df["Cement"].replace(0, np.nan)
            logger.info("WCR column created from Water / Cement")
        else:
            logger.warning("Missing 'WCR', 'Water', or 'Cement' columns → creating WCR with zeros")
            df["WCR"] = 0.0
    else:
        df["WCR"] = pd.to_numeric(df["WCR"], errors="coerce")

    # Fill NaNs before feature engineering
    df["Log_Age"] = df["Log_Age"].fillna(df["Log_Age"].median())
    df["WCR"] = df["WCR"].fillna(df["WCR"].median())

    # Feature engineering
    df["Log_Age_x_WCR"] = df["Log_Age"] * df["WCR"]

    # Build X strictly in model's expected order
    X = pd.DataFrame(index=df.index)
    for col in model.feature_names_in_:
        if col in df.columns:
            X[col] = df[col]
        else:
            logger.warning(f"Missing model feature '{col}' → filling with 0")
            X[col] = 0.0

    # Fill any remaining NaNs
    X.fillna(X.median(), inplace=True)
    X.fillna(0, inplace=True)

    return X[model.feature_names_in_]

# ---------------- MODEL ----------------
def load_model(path: str):
    model = joblib.load(path)
    logger.info("XGBoost model loaded successfully")
    logger.info(f"Expected feature order: {list(model.feature_names_in_)}")
    return model


# ---------------- INFERENCE HEALTH REPORT FOR MATERIAL STRENGTH ----------------
def generate_health_report(df: pd.DataFrame, pred_col: str):
    total = len(df)
    avg_strength = df[pred_col].mean()
    min_strength = df[pred_col].min()
    max_strength = df[pred_col].max()
    top_threshold = df[pred_col].quantile(0.95)  # top 5% strongest
    low_threshold = df[pred_col].quantile(0.05)  # bottom 5% weakest

    print("\n" + "="*50)
    print("MATERIAL STRENGTH INFERENCE HEALTH REPORT")
    print("="*50)
    print(f"Total Samples Scanned     : {total}")
    print(f"Average Predicted Strength: {avg_strength:.2f}")
    print(f"Minimum Predicted Strength: {min_strength:.2f}")
    print(f"Maximum Predicted Strength: {max_strength:.2f}")
    print("-" * 50)

    # Distribution Check
    print("Predicted Strength Distribution:")
    print(df[pred_col].describe().to_string())
    print("-" * 50)

    # Quick check for extreme predictions
    if max_strength > 100:  # assuming 100 MPa is unusually high
        print("ALERT: Some predicted strengths are unusually high. Check input features.")
    elif min_strength < 0:
        print("ALERT: Some predicted strengths are negative. Check input features.")
    else:
        print("Predicted strength values look within expected operational bounds.")
    print("="*50 + "\n")

    # Top strongest materials
    print("TOP 5 STRONGEST PREDICTIONS:")
    print(df.sort_values(by=pred_col, ascending=False)[[pred_col]].head(5).to_string(index=False))

    # Top weakest materials
    print("\nTOP 5 WEAKEST PREDICTIONS:")
    print(df.sort_values(by=pred_col, ascending=True)[[pred_col]].head(5).to_string(index=False))



# ---------------- PIPELINE ----------------
def run_pipeline():
    cfg = Config()
    model = load_model(cfg.MODEL_PATH)
    df_raw = fetch_databricks_data(cfg)
    logger.info(f"Fetched data shape: {df_raw.shape}")

    X = preprocess_features(df_raw, model)
    logger.info("Running inference...")
    df_raw["Predicted_Strength"] = np.expm1(model.predict(X))  # inverse log1p

    # Generate production-style health report
    generate_health_report(df_raw, "Predicted_Strength")

    # Save predictions
    df_raw.to_parquet("strength_predictions_final.parquet", index=False)
    logger.info("Predictions saved successfully")

# ---------------- RUN ----------------
run_pipeline()


MATERIAL STRENGTH INFERENCE HEALTH REPORT
Total Samples Scanned     : 1030
Average Predicted Strength: 31.48
Minimum Predicted Strength: 2.29
Maximum Predicted Strength: 78.70
--------------------------------------------------
Predicted Strength Distribution:
count    1030.000000
mean       31.480806
std        15.594826
min         2.291789
25%        19.265467
50%        29.743654
75%        41.627557
max        78.699738
--------------------------------------------------
Predicted strength values look within expected operational bounds.

TOP 5 STRONGEST PREDICTIONS:
 Predicted_Strength
          78.699738
          77.046188
          77.046188
          77.046188
          77.046188

TOP 5 WEAKEST PREDICTIONS:
 Predicted_Strength
           2.291789
           2.999516
           3.291507
           3.784400
           4.119390
